# Token Sequence Workbench

Run tokens one by one, inspect `State` after each step, evaluate forecasts, and prototype new tokens.

---
## Part 1 — Environment Setup

In [ ]:
import os, sys

%cd /content

if not os.path.exists('graph_Time_series'):
    !git clone https://github.com/chahineNejm/graph_Time_series
if not os.path.exists('kernels_playground'):
    !git clone https://github.com/chahineNejm/kernels_playground

for p in ['/content/graph_Time_series',
          '/content/kernels_playground',
          '/content/kernels_playground/first_tests']:
    if p not in sys.path:
        sys.path.insert(0, p)

!pip install -q datasets properscoring scikit-learn pandas lark

print('Ready.')

/content
Ready.


---
## Part 2 — Load Token Framework

In [ ]:
from pathlib import Path
import importlib.util
import types
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

PACKAGE_DIR = Path('/content/graph_Time_series/graph_Time_series')


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


pkg = types.ModuleType('graph_Time_series')
pkg.__path__ = [str(PACKAGE_DIR)]
sys.modules.setdefault('graph_Time_series', pkg)

blocks = types.ModuleType('graph_Time_series.token_blocks')
blocks.__path__ = [str(PACKAGE_DIR / 'token_blocks')]
sys.modules.setdefault('graph_Time_series.token_blocks', blocks)

artifacts_mod = load_module('graph_Time_series.artifacts', PACKAGE_DIR / 'artifacts.py')
token_mod = load_module('graph_Time_series.token', PACKAGE_DIR / 'token.py')
state_mod = load_module('graph_Time_series.state', PACKAGE_DIR / 'state.py')
bindings_mod = load_module('graph_Time_series.token_blocks.bindings',
                           PACKAGE_DIR / 'token_blocks' / 'bindings.py')
norm_mod  = load_module('graph_Time_series.token_blocks.normalization',
                        PACKAGE_DIR / 'token_blocks' / 'normalization.py')
rbf_mod   = load_module('graph_Time_series.token_blocks.kernel_rbf',
                        PACKAGE_DIR / 'token_blocks' / 'kernel_rbf.py')

State                   = state_mod.State
TransformRecord         = state_mod.TransformRecord
ArtifactSpec            = artifacts_mod.ArtifactSpec
InputBundle             = artifacts_mod.InputBundle
BindFeatureToken        = bindings_mod.BindFeatureToken
BindScaledHistoryToken  = bindings_mod.BindScaledHistoryToken
BindAllSafeTabularToken = bindings_mod.BindAllSafeTabularToken
StackFeatureBundleToken = bindings_mod.StackFeatureBundleToken
ZNormalizationToken     = norm_mod.ZNormalizationToken
KernelRBFToken          = rbf_mod.KernelRBFToken

Token          = token_mod.Token
TransformToken = token_mod.TransformToken
FeatureToken   = token_mod.FeatureToken
ModelToken     = token_mod.ModelToken

print('Loaded from', PACKAGE_DIR)

Loaded from /content/graph_Time_series/graph_Time_series


---
## Part 3 — Load Data

Uses `kernels_playground` utilities (same pattern as `tweaks.ipynb`).

In [ ]:
from utils.config import DATASETS
from utils.data import build_examples

DATA_CONFIG = 'electricity_H_long'
N_EXAMPLES_TO_DOWNLOAD = 2000
MAX_SAMPLES = 2000
HOLDOUT     = 20
HISTORY_LENGTH = 20_000   # keep only the last N history points
FUTURE_LENGTH  = 720  # None keeps the shortest available future length

raw = build_examples(
    config=DATA_CONFIG,
    start=0, stop=N_EXAMPLES_TO_DOWNLOAD, step=5,
    dataset_name=DATASETS['eval'],
)
print(f'Loaded {len(raw)} raw examples')

examples = raw[:MAX_SAMPLES]
min_hist = min(len(e['history']) for e in examples)
min_fut  = min(len(e['future'])  for e in examples)
history_len = min(HISTORY_LENGTH, min_hist) if HISTORY_LENGTH is not None else min_hist
future_len  = min(FUTURE_LENGTH, min_fut) if FUTURE_LENGTH is not None else min_fut

H_all = np.stack([e['history'][-history_len:] for e in examples]).astype(np.float32)
F_all = np.stack([e['future'][:future_len]   for e in examples]).astype(np.float32)

H, F                     = H_all[:-HOLDOUT], F_all[:-HOLDOUT]
H_holdout, F_holdout     = H_all[-HOLDOUT:], F_all[-HOLDOUT:]

print(f'Data config:   {DATA_CONFIG}')
print(f'Downloaded:    {len(raw)} examples')
print(f'Kept lengths:  history={history_len}  future={future_len}')
print(f'Run data:      H {H.shape}  F {F.shape}')
print(f'Held-out data: H {H_holdout.shape}  F {F_holdout.shape}')

Loaded 1850 raw examples
Data config:   electricity_H_long
Downloaded:    1850 examples
Kept lengths:  history=20000  future=720
Run data:      H (1830, 20000)  F (1830, 720)
Held-out data: H (20, 20000)  F (20, 720)


---
## Part 4 — State Inspector

`inspect(state)` — full snapshot of every field inside a State.
`diff(before, after)` — highlights what a single token changed.

In [ ]:
def inspect(state, label="State"):
    """Pretty-print every field in a State object."""
    print(f"\n{'=' * 65}")
    print(f"  {label}")
    print(f"{'=' * 65}")

    # Identity
    print(f"  tokens applied : {state.token_sequence}")
    print(f"  class counts   : {state.class_counts}")
    print(f"  terminated     : {state.terminated}")
    print(f"  depth          : {state.depth}")
    print(f"  n_models       : {state.n_models_applied}")
    print(f"  mase           : {state.mase}")
    print()

    # Core arrays
    print("  Core arrays:")
    for aname in ["original_history", "original_future",
                   "active_target_base", "current_target"]:
        arr = getattr(state, aname)
        print(f"    {aname:25s} {str(arr.shape):15s}  "
              f"range [{arr.min():.4f}, {arr.max():.4f}]  "
              f"mean {arr.mean():.4f}")
    print()

    # Feature stores
    def _show_store(title, d):
        if not d:
            print(f"  {title}: (empty)")
            return
        print(f"  {title}:")
        for k, v in d.items():
            if hasattr(v, "shape"):
                print(f"    {k:30s} {str(v.shape):15s}  "
                      f"range [{v.min():.4f}, {v.max():.4f}]")
            else:
                print(f"    {k:30s} {type(v).__name__}: {v}")

    _show_store("historical_features", state.historical_features)
    _show_store("future_features", state.future_features)
    _show_store("features (legacy)", state.features)
    print()

    # Metadata & flags
    if state.metadata:
        print("  metadata:")
        for k, v in state.metadata.items():
            vstr = f"array {v.shape}" if hasattr(v, "shape") else f"{v}"
            print(f"    {k}: {vstr}")
    if state.flags:
        print(f"  flags: {state.flags}")
    else:
        print("  flags: (empty)")
    print()

    # Transform stack
    if state.transform_stack:
        print(f"  Transform stack ({len(state.transform_stack)}):")
        for i, t in enumerate(state.transform_stack):
            inv = "yes" if t.inverse_fn else "no"
            print(f"    [{i}] {t.name}  (inverse: {inv}, affects: {t.affects})")
            if t.params:
                for k, v in t.params.items():
                    vstr = f"array {v.shape}" if hasattr(v, "shape") else f"{v}"
                    print(f"         {k}: {vstr}")
    else:
        print("  Transform stack: (empty)")
    print()

    # Prediction stack
    if state.prediction_stack:
        print(f"  Prediction stack ({len(state.prediction_stack)}):")
        for i, (pname, pred) in enumerate(
            zip(state.prediction_names, state.prediction_stack)
        ):
            print(f"    [{i}] {pname:20s} {str(pred.shape):15s}  "
                  f"range [{pred.min():.4f}, {pred.max():.4f}]  "
                  f"mean {pred.mean():.4f}")
    else:
        print("  Prediction stack: (empty)")
    print()

    if state.pending_conditions:
        print(f"  pending_conditions: {len(state.pending_conditions)} inverse fn(s)")
    print(f"  log entries: {len(state.log)}")
    print(f"{'=' * 65}\n")


def diff(before, after, label=""):
    """Show what changed between two State snapshots."""
    title = f"Diff: {label}" if label else "Diff"
    print(f"\n{'─' * 55}")
    print(f"  {title}")
    print(f"{'─' * 55}")

    new_tokens = after.token_sequence[len(before.token_sequence):]
    if new_tokens:
        print(f"  + tokens: {new_tokens}")

    for store_name in ("historical_features", "future_features"):
        old_keys = set(getattr(before, store_name).keys())
        new_keys = set(getattr(after, store_name).keys())
        for k in sorted(new_keys - old_keys):
            v = getattr(after, store_name)[k]
            print(f"  + {store_name}.{k}  {v.shape}")

    n_old = len(before.transform_stack)
    for t in after.transform_stack[n_old:]:
        print(f"  + transform: {t.name}  (affects: {t.affects})")

    n_old_pred = len(before.prediction_stack)
    for pname, pred in zip(
        after.prediction_names[n_old_pred:],
        after.prediction_stack[n_old_pred:],
    ):
        print(f"  + prediction: {pname}  {pred.shape}")

    if not np.array_equal(before.active_target_base, after.active_target_base):
        print(f"  ~ active_target_base changed")
    if not np.array_equal(before.current_target, after.current_target):
        resid_norm = np.linalg.norm(after.current_target)
        print(f"  ~ current_target changed  (||residual|| = {resid_norm:.4f})")

    new_flags = {k: v for k, v in after.flags.items() if k not in before.flags}
    if new_flags:
        print(f"  + flags: {new_flags}")
    print(f"{'─' * 55}\n")


print("inspect() and diff() ready.")

inspect() and diff() ready.


---
## Part 5 — Data Augmentation Token (example custom token)

A full working token that jitters + smooths the history before normalization.
Lives right here in the notebook — edit it freely.

In [ ]:
class DataAugmentationToken(TransformToken):
    """Augment the raw history with jitter and/or smoothing.

    Applies on original_history so downstream tokens
    (ZNormalization, models) see the augmented version.
    """

    name        = "DataAugmentation"
    token_class = "cleaning"
    max_uses    = 1
    reads       = ("raw_history",)
    writes      = ("raw_history", "augmented_history")
    description = "Jitter + smooth the history for robustness."

    def __init__(self, jitter_sigma=0.03, smooth_window=5, seed=42):
        self.jitter_sigma  = jitter_sigma
        self.smooth_window = smooth_window
        self.seed          = seed

    def check_specific_conditions(self, state):
        if state.flags.get("augmented", False):
            return False
        return super().check_specific_conditions(state)

    def apply(self, state):
        state = state.copy()
        rng  = np.random.default_rng(self.seed)
        hist = state.original_history.copy()   # (n_samples, T)

        # Jitter
        if self.jitter_sigma > 0:
            for i in range(hist.shape[0]):
                scale = self.jitter_sigma * np.std(hist[i])
                if scale < 1e-12:
                    scale = self.jitter_sigma
                hist[i] += rng.normal(0.0, scale, size=hist.shape[1])

        # Smooth
        if self.smooth_window > 1:
            w = self.smooth_window
            kernel = np.ones(w) / w
            for i in range(hist.shape[0]):
                pad_l = w // 2
                pad_r = w - 1 - pad_l
                padded = np.pad(hist[i], (pad_l, pad_r), mode="edge")
                hist[i] = np.convolve(padded, kernel, mode="valid")

        # Store
        state.original_history = hist
        state.features["raw_history"] = hist.copy()
        state.add_historical_feature("augmented_history", hist)

        state.register_transform(
            name=self.name,
            inverse_fn=None,
            params={
                "jitter_sigma":  self.jitter_sigma,
                "smooth_window": self.smooth_window,
                "seed":          self.seed,
            },
            affects="feature",
        )
        state.flags["augmented"] = True

        self._log_execution(
            state,
            reads={"original_history": state.original_history.shape},
            writes={"augmented_history": hist.shape},
        )
        return state


print("DataAugmentationToken defined.")

DataAugmentationToken defined.


---
## Part 5b — FLAIR-Inspired Tokens

Notebook-local tokens inspired by FLAIR: context windowing, period selection, period folding, shape/level decomposition, and fast kernel variants.

In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import pairwise_distances


FREQ_PERIODS = {
    'H': [24, 168],
    'D': [7, 365],
    'W': [52],
    'M': [12],
    '15T': [4, 96],
    '5T': [12, 288],
}


def median_lengthscale(X, max_points=24, seed=0):
    X = np.asarray(X, dtype=np.float32)
    if X.shape[0] <= 1:
        return 1.0
    rng = np.random.default_rng(seed)
    n = min(max_points, X.shape[0])
    idx = rng.choice(X.shape[0], size=n, replace=False)
    d2 = pairwise_distances(X[idx], metric='sqeuclidean')
    d = np.sqrt(d2[np.triu_indices(n, k=1)])
    d = d[np.isfinite(d) & (d > 0.0)]
    return float(np.median(d)) if d.size else 1.0


def gamma_from_lengthscale(lengthscale):
    return 1.0 / (2.0 * max(float(lengthscale), 1e-8) ** 2)


class ContextWindowToken(TransformToken):
    name = 'ContextWindow'
    token_class = 'cleaning'
    max_uses = 1
    reads = ('raw_history',)
    writes = ('raw_history', 'context_history')
    description = 'Keep only the last context_length history points.'

    def __init__(self, context_length=720):
        self.context_length = context_length

    def apply(self, state):
        state = state.copy()
        hist = state.original_history[:, -self.context_length:].copy()
        state.original_history = hist
        state.features['raw_history'] = hist.copy()
        state.add_historical_feature('context_history', hist)
        state.metadata['context_length'] = hist.shape[1]
        self._log_execution(state, reads={'raw_history': None}, writes={'context_history': hist.shape})
        return state


class PeriodSelectionToken(FeatureToken):
    name = 'PeriodSelection'
    reads = ('raw_history',)
    writes = ('period',)
    description = 'Choose a simple MDL/BIC period from frequency candidates.'

    def __init__(self, freq='H', max_series=8):
        self.freq = freq
        self.max_series = max_series

    def apply(self, state):
        hist = np.asarray(state.features['raw_history'], dtype=np.float32)
        candidates = [p for p in FREQ_PERIODS.get(self.freq, [1]) if hist.shape[1] // p >= 3]
        candidates = candidates or [1]
        subset = hist[:min(self.max_series, hist.shape[0])]
        null_vals = []
        for y in subset:
            T = len(y)
            rss_null = float(np.var(y) * T)
            null_vals.append(T * np.log(max(rss_null / max(T, 1), 1e-30)) + np.log(max(T, 2)))
        scores = {1: float(np.mean(null_vals))}

        for p in candidates:
            if p == 1:
                continue
            vals = []
            for y in subset:
                nc = len(y) // p
                mat = y[-(nc * p):].reshape(nc, p).T
                s = np.linalg.svd(mat, compute_uv=False)
                rss = float(np.sum(s[1:] ** 2)) if len(s) > 1 else 0.0
                T = nc * p
                bic = T * np.log(max(rss / max(T, 1), 1e-30)) + (p + nc - 1) * np.log(max(T, 2))
                vals.append(bic)
            scores[p] = float(np.mean(vals))
        period = min(scores, key=scores.get)
        state.metadata['period'] = int(period)
        state.metadata['period_scores'] = scores
        state.flags['period_selected'] = True
        self._log_execution(state, reads={'raw_history': hist.shape}, writes={'period': period})
        return state


class PeriodFoldToken(FeatureToken):
    name = 'PeriodFold'
    reads = ('raw_history', 'period')
    writes = ('period_matrix', 'level_series')
    description = 'Fold history into period phases and compute level sums.'

    def apply(self, state):
        hist = np.asarray(state.features['raw_history'], dtype=np.float32)
        P = int(state.metadata.get('period', 1))
        nc = hist.shape[1] // P
        usable = nc * P
        mat = hist[:, -usable:].reshape(hist.shape[0], nc, P).transpose(0, 2, 1)
        level = mat.sum(axis=1)
        state.add_historical_feature('period_matrix', mat)
        state.add_historical_feature('level_series', level)
        state.metadata['n_complete_periods'] = int(nc)
        self._log_execution(state, reads={'raw_history': hist.shape, 'period': P}, writes={'period_matrix': mat.shape, 'level_series': level.shape})
        return state


class ShapeLevelToken(FeatureToken):
    name = 'ShapeLevel'
    reads = ('period_matrix', 'level_series')
    writes = ('shape_vector',)
    description = 'Estimate a frozen within-period shape vector.'

    def __init__(self, shape_k=2):
        self.shape_k = shape_k

    def apply(self, state):
        mat = state.historical_features['period_matrix']
        K = min(self.shape_k, mat.shape[2])
        recent = mat[:, :, -K:]
        totals = recent.sum(axis=1, keepdims=True)
        shape = np.where(np.abs(totals) > 1e-8, recent / totals, 1.0 / mat.shape[1]).mean(axis=2)
        shape = shape / np.maximum(shape.sum(axis=1, keepdims=True), 1e-8)
        state.add_historical_feature('shape_vector', shape.astype(np.float32))
        state.metadata['shape_k'] = K
        self._log_execution(state, reads={'period_matrix': mat.shape}, writes={'shape_vector': shape.shape})
        return state


class ShapeNaiveToken(ModelToken):
    name = 'shape_naive'
    reads = ('shape_vector', 'level_series')
    writes = ('prediction_stack',)
    description = 'Forecast by repeating the last level through the frozen shape.'

    def get_model(self):
        return {'model': 'last_level_times_shape'}

    def apply(self, state):
        shape = state.historical_features['shape_vector']
        level = state.historical_features['level_series']
        P = shape.shape[1]
        h = state.horizon
        phases = np.arange(h) % P
        last_level = level[:, -1]
        pred = last_level[:, None] * shape[:, phases]
        state.push_prediction(pred.astype(np.float32), self.name)
        self._log_execution(state, reads={'shape_vector': shape.shape, 'level_series': level.shape}, writes={'prediction_stack[-1]': pred.shape})
        return state


class KernelRBFFastToken(ModelToken):
    name = 'kernel_rbf_fast'
    reads = ()
    writes = ('prediction_stack',)
    accepted_input_kinds = {'sequence_flat', 'tabular'}
    description = 'Fast in-sample KernelRidge using active bundle or scaled_history fallback.'

    def __init__(self, alpha=1e-2, median_subset=24, seed=0):
        self.alpha = alpha
        self.median_subset = median_subset
        self.seed = seed

    def get_model(self):
        return {'model': 'KernelRidge', 'kernel': 'rbf', 'mode': 'fit_once'}

    def check_specific_conditions(self, state):
        if not super().check_specific_conditions(state):
            return False
        try:
            _, _, bundle = self._resolve_model_input(state)
        except KeyError:
            return False
        return bundle is None or bundle.kind in self.accepted_input_kinds

    def _resolve_model_input(self, state):
        if 'model_input' in state.historical_features:
            bundle = state.active_input_bundle()
            if bundle is not None and bundle.kind not in self.accepted_input_kinds:
                raise ValueError(f'{self.name} cannot consume bundle kind {bundle.kind!r}')
            return np.asarray(state.historical_features['model_input'], dtype=np.float32), 'model_input', bundle
        if 'scaled_history' in state.historical_features:
            return np.asarray(state.historical_features['scaled_history'], dtype=np.float32), 'scaled_history', None
        raise KeyError('kernel_rbf_fast requires model_input or scaled_history')

    def apply(self, state):
        X, input_name, bundle = self._resolve_model_input(state)
        X = X.reshape(state.n_samples, -1)
        Y = np.asarray(state.current_target, dtype=np.float32)
        ell = median_lengthscale(X, self.median_subset, self.seed)
        gamma = gamma_from_lengthscale(ell)
        model = KernelRidge(kernel='rbf', alpha=self.alpha, gamma=gamma)
        model.fit(X, Y)
        pred = model.predict(X).astype(np.float32)
        state.push_prediction(pred, self.name)
        state.metadata[self.name] = {'lengthscale': ell, 'gamma': gamma, 'alpha': self.alpha,
                                     'input': input_name,
                                     'bundle': bundle.to_dict() if bundle else None}
        self._log_execution(state, reads={input_name: X.shape, 'current_target': Y.shape}, writes={'prediction_stack[-1]': pred.shape, 'lengthscale': ell})
        return state


class LevelKernelRBFToken(ModelToken):
    name = 'level_kernel_rbf'
    reads = ('level_series', 'shape_vector')
    writes = ('prediction_stack',)
    description = 'KernelRidge on FLAIR level series, expanded back through shape.'

    def __init__(self, alpha=1e-2, median_subset=24, seed=0):
        self.alpha = alpha
        self.median_subset = median_subset
        self.seed = seed

    def get_model(self):
        return {'model': 'KernelRidge', 'space': 'level'}

    def apply(self, state):
        X = np.asarray(state.historical_features['level_series'], dtype=np.float32)
        shape = np.asarray(state.historical_features['shape_vector'], dtype=np.float32)
        P = shape.shape[1]
        h = state.horizon
        m = int(np.ceil(h / P))
        y_level = np.zeros((state.n_samples, m), dtype=np.float32)
        for j in range(m):
            s = j * P
            e = min((j + 1) * P, h)
            y_level[:, j] = state.current_target[:, s:e].sum(axis=1)
        ell = median_lengthscale(X, self.median_subset, self.seed)
        gamma = gamma_from_lengthscale(ell)
        model = KernelRidge(kernel='rbf', alpha=self.alpha, gamma=gamma)
        model.fit(X, y_level)
        level_pred = model.predict(X).astype(np.float32)
        phases = np.arange(h) % P
        steps = np.arange(h) // P
        pred = level_pred[:, steps] * shape[:, phases]
        state.push_prediction(pred.astype(np.float32), self.name)
        state.metadata[self.name] = {'lengthscale': ell, 'gamma': gamma, 'alpha': self.alpha}
        self._log_execution(state, reads={'level_series': X.shape, 'shape_vector': shape.shape}, writes={'prediction_stack[-1]': pred.shape})
        return state


print('FLAIR-inspired tokens defined.')

---
## Part 6 — Token Registry

All tokens in one place. Add your own here.

In [ ]:
def make_tokens():
    return {
        'DataAugmentation': DataAugmentationToken(jitter_sigma=0.03, smooth_window=5),
        'ContextWindow':    ContextWindowToken(context_length=min(720, H.shape[1])),
        'PeriodSelection':  PeriodSelectionToken(freq='H'),
        'PeriodFold':       PeriodFoldToken(),
        'ShapeLevel':       ShapeLevelToken(shape_k=2),
        'shape_naive':      ShapeNaiveToken(),
        'ZNormalization':   ZNormalizationToken(),
        'BindScaledHistory': BindScaledHistoryToken(),
        'BindAllSafeTabular': BindAllSafeTabularToken(),
        'BindRawHistory':   BindFeatureToken('raw_history', token_name='BindRawHistory',
                                           bundle_name='raw_history', kind='sequence_flat'),
        'StackScaledContext': StackFeatureBundleToken(('scaled_history', 'context_history'),
                                                    token_name='StackScaledContext',
                                                    bundle_name='scaled_plus_context'),
        'kernel_rbf_fast':  KernelRBFFastToken(alpha=1e-2),
        'level_kernel_rbf': LevelKernelRBFToken(alpha=1e-2),
        'kernel_rbf_loo':   KernelRBFToken(),
    }


TOKENS = make_tokens()
print('Registered tokens:', list(TOKENS.keys()))

Registered tokens: ['DataAugmentation', 'ZNormalization', 'kernel_rbf']


---
## Part 6B - Bundle/Binder Architecture Playground

Feature tokens can leave many named artifacts in `State`. Binder tokens choose which artifacts become the active `model_input` bundle for generic model tokens. This cell stops before modeling so you can inspect the new architecture without reading target values.


In [ ]:
import pandas as pd
from IPython.display import display

def artifact_table(state):
    cols = ["name", "store", "kind", "role", "shape", "target_space", "source_token", "tags"]
    rows = state.describe_artifacts()
    return pd.DataFrame(rows)[cols] if rows else pd.DataFrame(columns=cols)

def bundle_table(state):
    cols = ["name", "kind", "artifact_names", "shape", "target_space", "source_token", "metadata"]
    rows = state.describe_input_bundles()
    return pd.DataFrame(rows)[cols] if rows else pd.DataFrame(columns=cols)

BUNDLE_SEQUENCE_LIBRARY = {
    "raw_history_bundle": ["BindRawHistory"],
    "explicit_scaled_bundle": ["ZNormalization", "BindScaledHistory"],
    "all_safe_tabular_bundle": ["ContextWindow", "ZNormalization", "BindAllSafeTabular"],
    "stack_scaled_context": ["ContextWindow", "ZNormalization", "StackScaledContext"],
}

def run_bundle_sequence(sequence, max_samples=48, verbose=True):
    tokens = make_tokens()
    st = State(H[:max_samples], F[:max_samples])
    if verbose:
        print("sequence:", " -> ".join(sequence))
    for name in sequence:
        tok = tokens[name]
        ok = tok.can_apply(st)
        if verbose:
            print(f"{name:22s} can_apply={ok}")
        if not ok:
            raise RuntimeError(f"{name} cannot apply after {st.token_sequence}")
        st = tok.apply(st)
    return st

# Change this name to inspect another staged combination.
BUNDLE_SEQUENCE_NAME = "explicit_scaled_bundle"
bundle_state = run_bundle_sequence(BUNDLE_SEQUENCE_LIBRARY[BUNDLE_SEQUENCE_NAME])

print("\nActive bundle:")
print(bundle_state.active_input_bundle())
print("\nArtifacts:")
display(artifact_table(bundle_state))
print("\nInput bundles:")
display(bundle_table(bundle_state))


---
## Part 7 — Define & Run Sequence

Edit `TOKEN_SEQUENCE`, then run. Full `inspect()` + `diff()` after every token.

In [ ]:
TOKEN_SEQUENCE = [
    'ContextWindow',
    'ZNormalization',
    'BindScaledHistory',
    'kernel_rbf_fast',
]

In [ ]:
state = State(H, F)
snapshots = {"init": state.copy()}
inspect(state, "INIT")

for i, name in enumerate(TOKEN_SEQUENCE):
    token = TOKENS[name]
    ok = token.can_apply(state)

    print(f"\n{'#' * 65}")
    print(f"  Step {i+1}/{len(TOKEN_SEQUENCE)}: {name}   can_apply = {ok}")
    print(f"{'#' * 65}")

    if not ok:
        print(f"  SKIPPED - token cannot apply in current state.")
        continue

    before = state.copy()
    state  = token.apply(state)
    snapshots[name] = state.copy()

    inspect(state, f"AFTER  {name}")
    diff(before, state, label=name)

print("\nAll snapshots:", list(snapshots.keys()))


  INIT
  tokens applied : []
  class counts   : {}
  terminated     : False
  depth          : 0
  n_models       : 0
  mase           : None

  Core arrays:
    original_history          (1830, 20000)    range [0.0000, 764000.0000]  mean 2299.3157
    original_future           (1830, 720)      range [0.0000, 672400.0000]  mean 2346.5659
    active_target_base        (1830, 720)      range [0.0000, 672400.0000]  mean 2346.5659
    current_target            (1830, 720)      range [0.0000, 672400.0000]  mean 2346.5659

  historical_features: (empty)
  future_features: (empty)
  features (legacy):
    raw_history                    (1830, 20000)    range [0.0000, 764000.0000]

  flags: (empty)

  Transform stack: (empty)

  Prediction stack: (empty)

  log entries: 0


#################################################################
  Step 1/3: DataAugmentation   can_apply = True
#################################################################

  AFTER  DataAugmentation
  tokens applie

---
## Part 8 — Evaluation: Forecast vs Reality

Compare the final forecast against the real future using MASE, CRPS, RMSE, and relRMSE.

In [ ]:
from utils.kernels import rmse, relative_rmse, mase

# CRPS via properscoring
try:
    from properscoring import crps_ensemble
    def crps_per_sample(y_true, y_pred):
        """CRPS for a deterministic forecast (degenerate ensemble of size 1)."""
        scores = []
        for i in range(y_true.shape[0]):
            # crps_ensemble expects (obs_scalar, ensemble_1d)
            s = np.mean([crps_ensemble(y_true[i, t], y_pred[i:i+1, t])
                         for t in range(y_true.shape[1])])
            scores.append(s)
        return np.array(scores)
    print("Using properscoring for CRPS.")
except ImportError:
    def crps_per_sample(y_true, y_pred):
        """Fallback: CRPS for deterministic forecast = MAE."""
        return np.mean(np.abs(y_true - y_pred), axis=1)
    print("properscoring not found, CRPS = MAE fallback.")


def evaluate(state, H, F, label=""):
    """Compute metrics, print table, plot forecast vs actual."""
    forecast = state.get_final_prediction()
    n = forecast.shape[0]

    rmses     = np.array([rmse(F[i], forecast[i]) for i in range(n)])
    rel_rmses = np.array([relative_rmse(F[i], forecast[i]) for i in range(n)])
    mases     = np.array([mase(F[i], forecast[i], H[i]) for i in range(n)])
    crps_vals = crps_per_sample(F, forecast)

    title = f"Evaluation: {label}" if label else "Evaluation"
    print(f"\n{'=' * 60}")
    print(f"  {title}   ({n} samples)")
    print(f"{'=' * 60}")
    print(f"  {'Metric':<18s} {'Mean':>10s} {'Median':>10s} {'Std':>10s}")
    print(f"  {'---' * 16}")
    for mname, vals in [("RMSE", rmses), ("relRMSE", rel_rmses),
                        ("MASE", mases), ("CRPS", crps_vals)]:
        print(f"  {mname:<18s} {vals.mean():10.4f} {np.median(vals):10.4f} {vals.std():10.4f}")
    print(f"{'=' * 60}\n")

    # Plot
    n_plot = min(6, n)
    fig, axes = plt.subplots(n_plot, 1, figsize=(12, 2.5 * n_plot), sharex=False)
    if n_plot == 1:
        axes = [axes]

    context = min(200, H.shape[1])
    for idx in range(n_plot):
        ax = axes[idx]
        h_ctx = H[idx, -context:]
        t_hist = np.arange(-len(h_ctx), 0)
        t_fut  = np.arange(0, F.shape[1])

        ax.plot(t_hist, h_ctx, color="steelblue", alpha=0.5, label="history")
        ax.plot(t_fut, F[idx], color="black", linewidth=1.5, label="actual")
        ax.plot(t_fut, forecast[idx], color="crimson", linewidth=1.5,
                linestyle="--", label="forecast")
        ax.axvline(0, color="gray", linestyle=":", alpha=0.5)
        ax.set_title(f"Sample {idx}  |  MASE={mases[idx]:.3f}  "
                     f"relRMSE={rel_rmses[idx]:.3f}  CRPS={crps_vals[idx]:.3f}",
                     fontsize=10)
        if idx == 0:
            ax.legend(fontsize=8, loc="upper left")

    plt.tight_layout()
    plt.show()

    return {"forecast": forecast, "rmse": rmses, "rel_rmse": rel_rmses,
            "mase": mases, "crps": crps_vals}


print("evaluate() ready.")

---
## Part 8b - Leakage-Safe Holdout Comparison

This section fits learned models only on training futures and predicts the held-out histories. The table is ranked by holdout metrics only. Cross-validation columns are optional diagnostics; in-sample/run metrics are intentionally not shown.

In [ ]:
import pandas as pd

TREE_MAX_SAMPLES = min(160, H.shape[0])
TREE_MAX_HOLDOUT = min(40, H_holdout.shape[0])
RUN_CV = True
CV_FOLDS = 3
CV_SEED = 7

H_tree, F_tree = H[:TREE_MAX_SAMPLES], F[:TREE_MAX_SAMPLES]
H_tree_holdout, F_tree_holdout = H_holdout[:TREE_MAX_HOLDOUT], F_holdout[:TREE_MAX_HOLDOUT]

TRAINING_ONLY_TOKENS = {'DataAugmentation'}
MODEL_TOKENS = {'kernel_rbf_fast', 'kernel_rbf_loo', 'shape_naive', 'level_kernel_rbf'}

CANDIDATE_SEQUENCES = {
    'plain__kernel_direct': ['ZNormalization', 'kernel_rbf_fast'],
    'plain__kernel_scaled_bundle': ['ZNormalization', 'BindScaledHistory', 'kernel_rbf_fast'],
    'context__kernel_all_safe': ['ContextWindow', 'ZNormalization', 'BindAllSafeTabular', 'kernel_rbf_fast'],
    'context__kernel_stack_scaled_context': ['ContextWindow', 'ZNormalization', 'StackScaledContext', 'kernel_rbf_fast'],
    'aug__kernel_scaled_bundle': ['DataAugmentation', 'ZNormalization', 'BindScaledHistory', 'kernel_rbf_fast'],
    'aug_context__kernel_all_safe': ['DataAugmentation', 'ContextWindow', 'ZNormalization', 'BindAllSafeTabular', 'kernel_rbf_fast'],
    'plain__flair_shape_naive': ['PeriodSelection', 'PeriodFold', 'ShapeLevel', 'shape_naive'],
    'context__flair_shape_naive': ['ContextWindow', 'PeriodSelection', 'PeriodFold', 'ShapeLevel', 'shape_naive'],
    'plain__flair_level_kernel': ['PeriodSelection', 'PeriodFold', 'ShapeLevel', 'level_kernel_rbf'],
    'context__flair_level_kernel': ['ContextWindow', 'PeriodSelection', 'PeriodFold', 'ShapeLevel', 'level_kernel_rbf'],
}

def strip_training_only(sequence):
    return [name for name in sequence if name not in TRAINING_ONLY_TOKENS]

def split_sequence(sequence):
    positions = [i for i, name in enumerate(sequence) if name in MODEL_TOKENS]
    if len(positions) != 1:
        raise ValueError(f'Expected exactly one model token, got {positions} for {sequence}')
    idx = positions[0]
    if idx != len(sequence) - 1:
        raise ValueError('This leakage-safe evaluator expects the model token to be last.')
    return sequence[:idx], sequence[idx]

def apply_feature_tokens(sequence, Hx, Fx, *, training):
    tokens = make_tokens()
    st = State(Hx, Fx)
    for name in sequence:
        if (not training) and name in TRAINING_ONLY_TOKENS:
            continue
        if name in MODEL_TOKENS:
            raise ValueError(f'Model token {name} reached while building features.')
        tok = tokens[name]
        if not tok.can_apply(st):
            raise RuntimeError(f'{name} cannot apply after {st.token_sequence}')
        st = tok.apply(st)
    return st

def rbf_input(state):
    if 'model_input' in state.historical_features:
        return np.asarray(state.historical_features['model_input'], dtype=np.float32).reshape(state.n_samples, -1), 'model_input'
    if 'scaled_history' in state.historical_features:
        return np.asarray(state.historical_features['scaled_history'], dtype=np.float32).reshape(state.n_samples, -1), 'scaled_history'
    if 'level_series' in state.historical_features:
        return np.asarray(state.historical_features['level_series'], dtype=np.float32).reshape(state.n_samples, -1), 'level_series'
    raise KeyError('No safe RBF input found. Add a binder or a feature token first.')

def push_safe_kernel_rbf(train_state, query_state, *, alpha=1e-2, median_subset=24, seed=0):
    X_train, input_name = rbf_input(train_state)
    X_query, _ = rbf_input(query_state)
    if X_train.shape[1] != X_query.shape[1]:
        raise ValueError(f'Feature width mismatch: train {X_train.shape}, query {X_query.shape}')
    Y_train = np.asarray(train_state.current_target, dtype=np.float32)
    ell = median_lengthscale(X_train, median_subset, seed)
    gamma = gamma_from_lengthscale(ell)
    model = KernelRidge(kernel='rbf', alpha=alpha, gamma=gamma)
    model.fit(X_train, Y_train)
    pred = model.predict(X_query).astype(np.float32)
    query_state.push_prediction(pred, 'safe_kernel_rbf')
    query_state.metadata['safe_kernel_rbf'] = {'input': input_name, 'lengthscale': ell, 'gamma': gamma, 'alpha': alpha, 'train_samples': int(X_train.shape[0])}
    return query_state

def push_safe_shape_naive(query_state):
    tok = ShapeNaiveToken()
    if not tok.can_apply(query_state):
        raise RuntimeError(f'shape_naive cannot apply after {query_state.token_sequence}')
    return tok.apply(query_state)

def push_safe_level_kernel(train_state, query_state, *, alpha=1e-2, median_subset=24, seed=0):
    X_train = np.asarray(train_state.historical_features['level_series'], dtype=np.float32)
    X_query = np.asarray(query_state.historical_features['level_series'], dtype=np.float32)
    if X_train.shape[1] != X_query.shape[1]:
        raise ValueError(f'Level feature width mismatch: train {X_train.shape}, query {X_query.shape}')
    shape_query = np.asarray(query_state.historical_features['shape_vector'], dtype=np.float32)
    P = shape_query.shape[1]
    h = query_state.horizon
    m = int(np.ceil(h / P))
    y_level = np.zeros((train_state.n_samples, m), dtype=np.float32)
    for j in range(m):
        start = j * P
        end = min((j + 1) * P, train_state.horizon)
        y_level[:, j] = train_state.current_target[:, start:end].sum(axis=1)
    ell = median_lengthscale(X_train, median_subset, seed)
    gamma = gamma_from_lengthscale(ell)
    model = KernelRidge(kernel='rbf', alpha=alpha, gamma=gamma)
    model.fit(X_train, y_level)
    level_pred = model.predict(X_query).astype(np.float32)
    phases = np.arange(h) % P
    steps = np.arange(h) // P
    pred = level_pred[:, steps] * shape_query[:, phases]
    query_state.push_prediction(pred.astype(np.float32), 'safe_level_kernel_rbf')
    query_state.metadata['safe_level_kernel_rbf'] = {'lengthscale': ell, 'gamma': gamma, 'alpha': alpha, 'period': int(P), 'train_samples': int(X_train.shape[0])}
    return query_state

def metric_summary_from_forecast(forecast, Hx, Fx):
    return {
        'MASE': float(np.mean([mase(Fx[i], forecast[i], Hx[i]) for i in range(Fx.shape[0])])),
        'RMSE': float(np.mean([rmse(Fx[i], forecast[i]) for i in range(Fx.shape[0])])),
        'relRMSE': float(np.mean([relative_rmse(Fx[i], forecast[i]) for i in range(Fx.shape[0])])),
    }

def safe_score_sequence(sequence, H_train, F_train, H_query, F_query):
    feature_seq, model_name = split_sequence(sequence)
    train_state = apply_feature_tokens(feature_seq, H_train, F_train, training=True)
    query_state = apply_feature_tokens(strip_training_only(feature_seq), H_query, F_query, training=False)
    if model_name in {'kernel_rbf_fast', 'kernel_rbf_loo'}:
        query_state = push_safe_kernel_rbf(train_state, query_state)
    elif model_name == 'shape_naive':
        query_state = push_safe_shape_naive(query_state)
    elif model_name == 'level_kernel_rbf':
        query_state = push_safe_level_kernel(train_state, query_state)
    else:
        raise ValueError(f'No leakage-safe predictor for {model_name}')
    forecast = query_state.get_final_prediction()
    return metric_summary_from_forecast(forecast, H_query, F_query), query_state

def cv_splits(n, n_folds=3, seed=0):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    folds = np.array_split(perm, n_folds)
    for fold_idx, val_idx in enumerate(folds):
        train_idx = np.setdiff1d(np.arange(n), val_idx, assume_unique=False)
        yield fold_idx, train_idx, val_idx

SAFE_RESULTS = []
SAFE_STATES = {}
for label, seq in CANDIDATE_SEQUENCES.items():
    print(f'\nScoring {label}: {" -> ".join(seq)}')
    row = {'name': label, 'sequence': ' -> '.join(seq)}
    try:
        if RUN_CV:
            fold_rows = []
            for fold_idx, train_idx, val_idx in cv_splits(len(H_tree), CV_FOLDS, CV_SEED):
                metrics, _ = safe_score_sequence(seq, H_tree[train_idx], F_tree[train_idx], H_tree[val_idx], F_tree[val_idx])
                fold_rows.append(metrics)
            for metric_name in ('MASE', 'RMSE', 'relRMSE'):
                vals = np.array([r[metric_name] for r in fold_rows], dtype=float)
                row[f'cv_{metric_name}_mean'] = float(vals.mean())
                row[f'cv_{metric_name}_std'] = float(vals.std())
        holdout_metrics, holdout_state = safe_score_sequence(seq, H_tree, F_tree, H_tree_holdout, F_tree_holdout)
        SAFE_STATES[label] = holdout_state
        row.update({f'holdout_{k}': v for k, v in holdout_metrics.items()})
        row['active_bundle'] = (holdout_state.active_input_bundle().name if holdout_state.active_input_bundle() else '')
        print({k: row[k] for k in row if k.startswith('holdout_') or k == 'active_bundle'})
    except Exception as e:
        row['error'] = str(e)
        print('ERROR:', e)
    SAFE_RESULTS.append(row)

SAFE_RESULTS_DF = pd.DataFrame(SAFE_RESULTS)
if 'holdout_MASE' in SAFE_RESULTS_DF:
    SAFE_RESULTS_DF = SAFE_RESULTS_DF.sort_values('holdout_MASE', na_position='last')
DISPLAY_COLUMNS = ['name', 'sequence', 'active_bundle', 'holdout_MASE', 'holdout_relRMSE', 'holdout_RMSE', 'cv_MASE_mean', 'cv_MASE_std', 'cv_relRMSE_mean', 'cv_relRMSE_std', 'error']
DISPLAY_COLUMNS = [c for c in DISPLAY_COLUMNS if c in SAFE_RESULTS_DF.columns]
SAFE_RESULTS_DF[DISPLAY_COLUMNS]

### Plot Best Leakage-Safe Holdout Candidate

Uses the already-fitted holdout state from the safe comparison above.

In [ ]:
best_name = SAFE_RESULTS_DF.dropna(subset=['holdout_MASE']).iloc[0]['name']
best_seq = CANDIDATE_SEQUENCES[best_name]
print('Best holdout candidate:', best_name, best_seq)

best_state = SAFE_STATES[best_name]
best_metrics = evaluate(best_state, H_tree_holdout, F_tree_holdout, label=f'leakage-safe holdout: {best_name}')

print('Active bundle:', best_state.active_input_bundle())
print('Artifacts:')
display(artifact_table(best_state))
print('Input bundles:')
display(bundle_table(best_state))

In [ ]:
# Optional diagnostic only: this is in-sample for the manual step-through state.
metrics = evaluate(state, H, F, label='manual sequence in-sample diagnostic')

### Evaluate Current Sequence on Held-Out Data (Leakage-Safe)

In [ ]:
# Fit any learned model on run data only, then predict held-out histories.
safe_metrics_ho, state_ho = safe_score_sequence(TOKEN_SEQUENCE, H, F, H_holdout, F_holdout)
print('Leakage-safe holdout metrics:', safe_metrics_ho)
metrics_ho = evaluate(state_ho, H_holdout, F_holdout, label='current sequence leakage-safe holdout')
print('Active bundle:', state_ho.active_input_bundle())

---
## True Holdout Kernel Evaluation

Fits kernel regression on run data only, predicts held-out histories, then uses held-out futures only for scoring.

In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import pairwise_distances


def normalize_history_future_by_history(Hx, Fx):
    mu = Hx.mean(axis=1, keepdims=True)
    sigma = Hx.std(axis=1, keepdims=True)
    sigma = np.where(sigma < 1e-8, 1.0, sigma)
    return (Hx - mu) / sigma, (Fx - mu) / sigma, mu, sigma


def normalize_history_only(Hx):
    mu = Hx.mean(axis=1, keepdims=True)
    sigma = Hx.std(axis=1, keepdims=True)
    sigma = np.where(sigma < 1e-8, 1.0, sigma)
    return (Hx - mu) / sigma, mu, sigma


def true_holdout_kernel_rbf(
    H_train,
    F_train,
    H_test,
    F_test,
    context_length=720,
    alpha=1e-2,
    median_subset=64,
    seed=0,
):
    # Context window uses history only.
    context = min(context_length, H_train.shape[1], H_test.shape[1])
    Htr = H_train[:, -context:].astype(np.float32)
    Hte = H_test[:, -context:].astype(np.float32)

    # Normalize each series using its own history only.
    X_train, Y_train, train_mu, train_sigma = normalize_history_future_by_history(Htr, F_train)
    X_test, test_mu, test_sigma = normalize_history_only(Hte)

    X_train = X_train.reshape(X_train.shape[0], -1)
    X_test = X_test.reshape(X_test.shape[0], -1)

    ell = median_lengthscale(X_train, max_points=median_subset, seed=seed)
    gamma = gamma_from_lengthscale(ell)

    model = KernelRidge(kernel='rbf', alpha=alpha, gamma=gamma)
    model.fit(X_train, Y_train)

    pred_norm = model.predict(X_test).astype(np.float32)
    pred = pred_norm * test_sigma + test_mu

    metrics = {
        'MASE': float(np.mean([mase(F_test[i], pred[i], H_test[i]) for i in range(F_test.shape[0])])),
        'RMSE': float(np.mean([rmse(F_test[i], pred[i]) for i in range(F_test.shape[0])])),
        'relRMSE': float(np.mean([relative_rmse(F_test[i], pred[i]) for i in range(F_test.shape[0])])),
        'lengthscale': ell,
        'gamma': gamma,
        'alpha': alpha,
        'context': context,
    }
    return pred, metrics, model


true_holdout_pred, true_holdout_metrics, true_holdout_model = true_holdout_kernel_rbf(
    H, F, H_holdout, F_holdout,
    context_length=min(720, H.shape[1]),
    alpha=1e-2,
    median_subset=64,
    seed=0,
)

true_holdout_metrics

In [ ]:
# Plot true holdout kernel forecast vs actual.
n_plot = min(6, H_holdout.shape[0])
fig, axes = plt.subplots(n_plot, 1, figsize=(12, 2.5 * n_plot), sharex=False)
if n_plot == 1:
    axes = [axes]

context = min(200, H_holdout.shape[1])
for idx in range(n_plot):
    ax = axes[idx]
    h_ctx = H_holdout[idx, -context:]
    t_hist = np.arange(-len(h_ctx), 0)
    t_fut = np.arange(F_holdout.shape[1])
    sample_mase = mase(F_holdout[idx], true_holdout_pred[idx], H_holdout[idx])
    sample_rel = relative_rmse(F_holdout[idx], true_holdout_pred[idx])

    ax.plot(t_hist, h_ctx, color='steelblue', alpha=0.5, label='history')
    ax.plot(t_fut, F_holdout[idx], color='black', linewidth=1.5, label='actual')
    ax.plot(t_fut, true_holdout_pred[idx], color='darkorange', linestyle='--', linewidth=1.5, label='true holdout kernel')
    ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'Sample {idx} | MASE={sample_mase:.3f} | relRMSE={sample_rel:.3f}', fontsize=10)
    if idx == 0:
        ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

---
## Part 9 — Execution Log

In [ ]:
state.print_log()

---
## Part 10 — Re-inspect Any Snapshot

Change the key to look at state after any token.

In [ ]:
# Pick any snapshot
inspect(snapshots['ZNormalization'], 'snapshot: ZNormalization')

In [ ]:
# Compare any two snapshots
diff(snapshots['init'], snapshots['DataAugmentation'], label='init -> DataAugmentation')

---
---
## Part 11 — New Token Scratch Space

Prototype new tokens here.

**Base classes available:**
- `TransformToken` — modifies target/features (runs before models)
- `FeatureToken` — adds features to the state
- `ModelToken` — fits a model and pushes predictions
- `Token` — raw base, no constraints

Minimal template below.

In [ ]:
# Token template - uncomment and edit

# class MyNewToken(TransformToken):
#     name        = 'my_new_token'
#     token_class = 'transform'
#     max_uses    = 1
#     reads       = ('raw_history',)
#     writes      = ('my_feature',)
#     description = 'One-line description.'
#
#     def check_specific_conditions(self, state):
#         return 'my_flag' not in state.flags
#
#     def apply(self, state):
#         state = state.copy()
#         # Your logic here
#         # feat = some_computation(state.original_history)
#         # state.add_historical_feature('my_feature', feat)
#         # state.flags['my_flag'] = True
#         self._log_execution(state,
#             reads={'original_history': state.original_history.shape},
#             writes={'my_feature': 'computed'})
#         return state

In [ ]:
# Quick test on a small batch

# tok = MyNewToken()
# test_state = State(H[:4], F[:4])
# print('can_apply:', tok.can_apply(test_state))
# result = tok.apply(test_state)
# inspect(result, 'after MyNewToken')

In [ ]:
# Register and re-run

# TOKENS['my_new_token'] = MyNewToken()
# TOKEN_SEQUENCE.append('my_new_token')
# # Re-run Part 7 cells above